<a href="https://colab.research.google.com/github/Skquark/AEI-Colab-Notebooks/blob/main/Make-It-Animatable_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

View this notebook on GitHub: <a href="https://github.com/Skquark/AEI-Colab-Notebooks/blob/main/Make-It-Animatable_Colab.ipynb" target="_parent">Skquark/AEI-Colab-Notebooks/Make-It-Animatable_Colab.ipynb</a>


# 💃 Make-It-Animatable (MIA v2 & v1) — Sub-Second 3D Character & 3DGS Auto-Rigging, Skinning & Mixamo Motion Retargeting Studio (MIT)

Self-contained Google Colab studio for **[`jasongzy/Make-It-Animatable`](https://github.com/jasongzy/Make-It-Animatable)** (*"Make-It-Animatable: An Efficient Framework for Authoring Animation-Ready 3D Characters"*, **CVPR 2025 Highlight**, [arXiv 2411.18197](https://arxiv.org/abs/2411.18197), **MIT License**), featuring both **MIA v2** (upgraded **Hunyuan3D-2.1 ShapeVAE** backbone) and **MIA v1** (supporting **3D Gaussian Splatting `.ply`** characters).

Takes any static 3D humanoid/character mesh (`.glb`, `.obj`, `.fbx`, `.ply`) from `TRELLIS-2`, `Hunyuan3D-2.1`, `Hunyuan3D-3`, `TripoSplat`, or `Mesh_Optimizer` and generates a **Mixamo-compatible 52-bone skeleton, topological k-ring filtered skinning weights, canonical T-pose normalization, and retargeted `.glb` + `.fbx` animations in under 1 second of neural inference!**

---

### How It Complements Our Other 3D & Rigging Notebooks

| Notebook | Input | Output | Primary Strength |
| :--- | :--- | :--- | :--- |
| **[`Make-It-Animatable_Colab.ipynb`](https://github.com/Skquark/AEI-Colab-Notebooks/blob/main/Make-It-Animatable_Colab.ipynb) (This Notebook)** | 3D character mesh (`.glb`/`.obj`/`.ply`) **or** 3DGS `.ply` | **Animated `.glb` + `.fbx` + `.blend`** with Mixamo motion retargeting | **<1s feed-forward rigging & skinning** (`Hunyuan3D-2.1 ShapeVAE` in MIA v2), automatic **Pose-to-Rest (T-pose) straightening**, 52 Mixamo bones (including fingers), and instant **Mixamo `.fbx` motion retargeting**! |
| [`SkinTokens_Colab.ipynb`](https://github.com/Skquark/AEI-Colab-Notebooks/blob/main/SkinTokens_Colab.ipynb) | Arbitrary 3D mesh (`.glb`/`.obj`/`.fbx`) | Static rigged `.glb` (bones + weights) | Template-free autoregressive rigging (`Qwen3-0.6B` + FSQ-CVAE) for arbitrary non-humanoid creatures, quadrupeds, insects, and props. |
| [`LHM_Colab.ipynb`](https://github.com/Skquark/AEI-Colab-Notebooks/blob/main/LHM_Colab.ipynb) | 1–8 character photos | Animatable 3DGS `.ply` + SMPL-X video | Direct photo-to-3DGS human reconstruction (`LHMPP-700M`). |

---

### Architecture & 4-Stage Pipeline

```
Static 3D Character Mesh (.glb / .obj / .ply) or 3DGS (.ply)
        │
        ▼
  [Stage 1: Coarse Joint Localization (joints_coarse.pth)]
   • Centers Hips origin & aligns pelvic plane
   • Hierarchical surface + 50% high-density hand point sampling (N = 32,768)
        │
        ▼
  [Stage 2: Fine Joints, Skinning Weights & Pose-to-Rest (bw_joints.pth + pose.pth)]
   • Hunyuan3D-2.1 ShapeVAE backbone (MIA v2) with 100k-vertex chunked evaluation
   • k-ring sparse adjacency voting removes cross-limb blend-weight bleeding
   • ortho6d Pose Transformer straightens A-pose / action poses into Mixamo T-Pose
        │
        ▼
  [Stage 3: Headless Blender (bpy) + Auto-Rig Retargeting + FBX2glTF]
   • Binds 52-bone (or 22-bone finger-free) Mixamo armature & vertex groups
   • Retargets any Mixamo .fbx animation (in-place or root-motion) -> .fbx + .glb
```

* **License**: **MIT License** (`jasongzy/Make-It-Animatable`).
* **VRAM**: **~4–6 GB VRAM** (runs effortlessly on free Colab **T4 16 GB**, **L4**, or **A100**).


In [ ]:
# @title STEP 1 — Install Dependencies, Blender (`bpy`), `FBX2glTF` & Clone Repository
# @markdown Mounts Google Drive, installs `bpy` (with automatic `cp312` wheel fallback), `trimesh`, `potpourri3d`, `torch-cluster`, `pytorch3d` (with pure-PyTorch CUDA fallback shims so version mismatches never break execution), and clones `jasongzy/Make-It-Animatable` (`v2` branch).

import os, sys, time, subprocess, shutil, pathlib

t0 = time.time()
print("=" * 72)
print("💃 Make-It-Animatable (MIA v2 & v1) — Step 1: Environment & Repo Setup")
print("=" * 72)

MOUNT_DRIVE = True  # @param {type:"boolean"}
DRIVE_CACHE_DIR = "/content/drive/MyDrive/AEI_3D_Cache/Make_It_Animatable"  # @param {type:"string"}
OUTPUT_BASE_DIR = "/content/drive/MyDrive/AEI_3D_Out/Make-It-Animatable"  # @param {type:"string"}

if MOUNT_DRIVE:
    try:
        from google.colab import drive
        if not os.path.isdir("/content/drive/MyDrive"):
            drive.mount("/content/drive", force_remount=False)
        print("[Drive] Mounted at /content/drive/MyDrive")
    except ImportError:
        print("[Drive] Not running on Colab — using local directories.")
        DRIVE_CACHE_DIR = os.path.abspath("./_mia_cache")
        OUTPUT_BASE_DIR = os.path.abspath("./_mia_outputs")
    except Exception as e:
        print(f"[Drive] Mount warning ({e}) — falling back to /content/_mia_cache")
        DRIVE_CACHE_DIR = "/content/_mia_cache"
        OUTPUT_BASE_DIR = "/content/mia_outputs"
else:
    DRIVE_CACHE_DIR = "/content/_mia_cache"
    OUTPUT_BASE_DIR = "/content/mia_outputs"

os.makedirs(DRIVE_CACHE_DIR, exist_ok=True)
os.makedirs(OUTPUT_BASE_DIR, exist_ok=True)
os.environ["HF_HOME"] = os.path.join(DRIVE_CACHE_DIR, "hf_home")
os.environ["HUGGINGFACE_HUB_CACHE"] = os.path.join(DRIVE_CACHE_DIR, "hf_home", "hub")
os.makedirs(os.environ["HUGGINGFACE_HUB_CACHE"], exist_ok=True)

MIA_REPO_DIR = "/content/Make-It-Animatable"

# 1. Install core Python packages
base_pkgs = [
    "einops", "timm", "PyMCubes", "plyfile", "potpourri3d", "shapely",
    "trimesh[easy]", "scikit-image", "omegaconf", "transformers>=4.46.0",
    "accelerate", "diffusers", "huggingface_hub", "scipy", "matplotlib",
    "tqdm==4.66.5", "gradio==5.49.1"
]
print("[1/5] Installing core 3D geometry & deep learning packages...")
try:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet"] + base_pkgs, check=True)
except Exception as e:
    print(f"[pip] Batch install warning ({e}); installing packages individually...")
    for p in base_pkgs:
        subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", p], check=False)

# 2. Install torch-cluster & pytorch3d (with pure-PyTorch CUDA fallback shims if wheel unavailable)
import torch
pt_ver = torch.__version__.split("+")[0]
cu_ver = "cu" + (torch.version.cuda.replace(".", "") if torch.version.cuda else "cpu")
print(f"[2/5] Checking torch_cluster & pytorch3d for PyTorch {pt_ver}+{cu_ver}...")

try:
    import torch_cluster  # noqa: F401
    print("  • torch_cluster already installed.")
except ImportError:
    tc_res = subprocess.run(
        [sys.executable, "-m", "pip", "install", "--quiet", "torch-cluster", "-f", f"https://data.pyg.org/whl/torch-{pt_ver}+{cu_ver}.html"],
        capture_output=True, text=True, check=False
    )
    if tc_res.returncode != 0:
        print("  • Prebuilt torch-cluster wheel not found for this exact Torch+CUDA pair; installing pure-PyTorch CUDA FPS shim...")
        site_pkgs = next(p for p in sys.path if "site-packages" in p or "dist-packages" in p)
        tc_shim_dir = os.path.join(site_pkgs, "torch_cluster")
        os.makedirs(tc_shim_dir, exist_ok=True)
        with open(os.path.join(tc_shim_dir, "__init__.py"), "w", encoding="utf-8") as f:
            f.write('''import torch

def fps(src: torch.Tensor, batch: torch.Tensor = None, ratio: float = 0.5, random_start: bool = True) -> torch.Tensor:
    # Fast vectorized GPU Farthest Point Sampling fallback when compiled torch_cluster wheel is absent.
    device = src.device
    if batch is None:
        batch = torch.zeros(src.size(0), dtype=torch.long, device=device)
    deg = torch.bincount(batch)
    ptr = torch.cat([deg.new_zeros(1), deg.cumsum(dim=0)])
    out_indices = []
    for b in range(len(deg)):
        s_idx, e_idx = int(ptr[b].item()), int(ptr[b + 1].item())
        N = e_idx - s_idx
        if N == 0:
            continue
        k = max(1, min(N, int(round(N * float(ratio)))))
        pts = src[s_idx:e_idx]
        selected = torch.empty(k, dtype=torch.long, device=device)
        cur = torch.randint(0, N, (1,), device=device).item() if random_start else 0
        selected[0] = cur
        dists = torch.sum((pts - pts[cur : cur + 1]) ** 2, dim=-1)
        for i in range(1, k):
            cur = int(torch.argmax(dists).item())
            selected[i] = cur
            new_d = torch.sum((pts - pts[cur : cur + 1]) ** 2, dim=-1)
            dists = torch.minimum(dists, new_d)
        out_indices.append(selected + s_idx)
    return torch.cat(out_indices, dim=0) if out_indices else torch.empty(0, dtype=torch.long, device=device)
''')
        print("  • Installed torch_cluster.fps GPU shim.")

try:
    import pytorch3d.transforms  # noqa: F401
    print("  • pytorch3d already installed.")
except ImportError:
    p3d_res = subprocess.run(
        [sys.executable, "-m", "pip", "install", "--quiet", "--extra-index-url", "https://miropsota.github.io/torch_packages_builder", f"pytorch3d==0.7.8+pt{pt_ver}{cu_ver}"],
        capture_output=True, text=True, check=False
    )
    if p3d_res.returncode != 0:
        # Try git install without CUDA compilation (pure python transforms module) or create transforms shim
        site_pkgs = next(p for p in sys.path if "site-packages" in p or "dist-packages" in p)
        p3d_dir = os.path.join(site_pkgs, "pytorch3d")
        os.makedirs(p3d_dir, exist_ok=True)
        with open(os.path.join(p3d_dir, "__init__.py"), "w", encoding="utf-8") as f:
            f.write("# pytorch3d lightweight transforms shim\n")
        with open(os.path.join(p3d_dir, "transforms.py"), "w", encoding="utf-8") as f:
            f.write('''import torch
import torch.nn.functional as F

class Transform3d:
    def __init__(self, matrix: torch.Tensor = None, dtype: torch.dtype = torch.float32, device="cpu"):
        if matrix is None:
            self._matrix = torch.eye(4, dtype=dtype, device=device).unsqueeze(0)
        else:
            self._matrix = matrix if matrix.ndim == 3 else matrix.unsqueeze(0)

    def get_matrix(self) -> torch.Tensor:
        return self._matrix

    def to(self, *args, **kwargs) -> "Transform3d":
        return Transform3d(matrix=self._matrix.to(*args, **kwargs))

    def compose(self, *others: "Transform3d") -> "Transform3d":
        mat = self._matrix
        for o in others:
            omat = o.get_matrix().to(mat)
            mat = torch.bmm(mat.expand(max(len(mat), len(omat)), 4, 4), omat.expand(max(len(mat), len(omat)), 4, 4))
        return Transform3d(matrix=mat)

    def inverse(self, invert_composed: bool = False) -> "Transform3d":
        return Transform3d(matrix=torch.linalg.inv(self._matrix))

    def transform_points(self, points: torch.Tensor, eps: float | None = None) -> torch.Tensor:
        was_2d = (points.ndim == 2)
        pts = points.unsqueeze(0) if was_2d else points
        mat = self._matrix.to(pts)
        R = mat[:, :3, :3]
        T = mat[:, 3:, :3]
        out = torch.bmm(pts, R.expand(len(pts), 3, 3)) + T.expand(len(pts), 1, 3)
        return out.squeeze(0) if was_2d else out

    def transform_normals(self, normals: torch.Tensor) -> torch.Tensor:
        was_2d = (normals.ndim == 2)
        nrm = normals.unsqueeze(0) if was_2d else normals
        mat = self._matrix.to(nrm)
        R_inv_t = torch.linalg.inv(mat[:, :3, :3]).transpose(1, 2)
        out = torch.bmm(nrm, R_inv_t.expand(len(nrm), 3, 3))
        return out.squeeze(0) if was_2d else out

class Scale(Transform3d):
    def __init__(self, x, y=None, z=None, dtype=torch.float32, device="cpu"):
        if isinstance(x, torch.Tensor) and x.shape[-1] == 3:
            xyz = x if x.ndim == 2 else x.unsqueeze(0)
        else:
            s = float(x) if not isinstance(x, torch.Tensor) else float(x.item())
            sy = s if y is None else float(y)
            sz = s if z is None else float(z)
            xyz = torch.tensor([[s, sy, sz]], dtype=dtype, device=device)
        mat = torch.eye(4, dtype=xyz.dtype, device=xyz.device).unsqueeze(0).repeat(len(xyz), 1, 1)
        mat[:, 0, 0] = xyz[:, 0]
        mat[:, 1, 1] = xyz[:, 1]
        mat[:, 2, 2] = xyz[:, 2]
        super().__init__(matrix=mat)

class Translate(Transform3d):
    def __init__(self, x, y=None, z=None, dtype=torch.float32, device="cpu"):
        if isinstance(x, torch.Tensor) and x.shape[-1] == 3:
            xyz = x if x.ndim == 2 else x.unsqueeze(0)
        else:
            xyz = torch.tensor([[float(x), float(y or 0.0), float(z or 0.0)]], dtype=dtype, device=device)
        mat = torch.eye(4, dtype=xyz.dtype, device=xyz.device).unsqueeze(0).repeat(len(xyz), 1, 1)
        mat[:, 3, :3] = xyz
        super().__init__(matrix=mat)

class Rotate(Transform3d):
    def __init__(self, R: torch.Tensor, dtype=torch.float32, device="cpu"):
        if R.ndim == 2:
            R = R.unsqueeze(0)
        mat = torch.eye(4, dtype=R.dtype, device=R.device).unsqueeze(0).repeat(len(R), 1, 1)
        mat[:, :3, :3] = R
        super().__init__(matrix=mat)

def rotation_6d_to_matrix(d6: torch.Tensor) -> torch.Tensor:
    a1, a2 = d6[..., :3], d6[..., 3:]
    b1 = F.normalize(a1, dim=-1)
    b2 = a2 - (b1 * a2).sum(-1, keepdim=True) * b1
    b2 = F.normalize(b2, dim=-1)
    b3 = torch.cross(b1, b2, dim=-1)
    return torch.stack((b1, b2, b3), dim=-2)

def matrix_to_rotation_6d(matrix: torch.Tensor) -> torch.Tensor:
    batch_dim = matrix.size()[:-2]
    return matrix[..., :2, :].clone().reshape(batch_dim + (6,))

def quaternion_to_matrix(quaternions: torch.Tensor) -> torch.Tensor:
    r, i, j, k = torch.unbind(quaternions, -1)
    two_s = 2.0 / (quaternions * quaternions).sum(-1)
    o = torch.stack((
        1 - two_s * (j * j + k * k), two_s * (i * j - k * r), two_s * (i * k + j * r),
        two_s * (i * j + k * r), 1 - two_s * (i * i + k * k), two_s * (j * k - i * r),
        two_s * (i * k - j * r), two_s * (j * k + i * r), 1 - two_s * (i * i + j * j),
    ), -1)
    return o.reshape(quaternions.shape[:-1] + (3, 3))
''')
        print("  • Installed pytorch3d.transforms GPU shim.")

# 3. Install Blender Python module (bpy) with cp312 wheel fallback
print("[3/5] Ensuring Blender Python API (bpy) is installed...")
def _ensure_bpy_installed():
    try:
        import bpy  # noqa: F401
        print("  • bpy already importable.")
        return
    except Exception:
        pass
    for bpy_ver in ["bpy==4.3.0", "bpy==4.2.0"]:
        r = subprocess.run(
            [sys.executable, "-m", "pip", "install", "--quiet", "--extra-index-url", "https://download.blender.org/pypi/", bpy_ver],
            capture_output=True, text=True, check=False
        )
        if r.returncode == 0:
            try:
                import bpy  # noqa: F401
                print(f"  • Installed {bpy_ver} from Blender PyPI.")
                return
            except Exception:
                pass
    # Fallback for cp312 Colab runtimes: download cached cp312 wheel from VAST-AI/SkinTokens Space
    bpy_wheel = os.path.join(DRIVE_CACHE_DIR, "bpy-4.5.4rc0-cp312-cp312-manylinux_2_39_x86_64.whl")
    if not os.path.isfile(bpy_wheel) or os.path.getsize(bpy_wheel) < 1_000_000:
        from huggingface_hub import hf_hub_download
        dl_w = hf_hub_download(
            repo_id="VAST-AI/SkinTokens", repo_type="space",
            filename="bpy-4.5.4rc0-cp312-cp312-manylinux_2_39_x86_64.whl",
            local_dir=DRIVE_CACHE_DIR,
        )
        if os.path.abspath(dl_w) != os.path.abspath(bpy_wheel):
            shutil.move(dl_w, bpy_wheel)
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", bpy_wheel], check=True)
    print("  • Installed bpy 4.5.4rc0 (cp312 wheel).")

_ensure_bpy_installed()

# 4. Clone jasongzy/Make-It-Animatable (v2 branch with submodules)
print("[4/5] Cloning jasongzy/Make-It-Animatable (v2 branch)...")
if not os.path.isdir(os.path.join(MIA_REPO_DIR, ".git")):
    subprocess.run(
        ["git", "clone", "--recursive", "--single-branch", "--branch", "v2", "https://github.com/jasongzy/Make-It-Animatable.git", MIA_REPO_DIR],
        check=True
    )
else:
    subprocess.run(["git", "-C", MIA_REPO_DIR, "submodule", "update", "--init", "--recursive"], check=False)

# Create spaces stub if not present
try:
    import spaces  # noqa: F401
except ImportError:
    site_pkgs = next(p for p in sys.path if "site-packages" in p or "dist-packages" in p)
    with open(os.path.join(site_pkgs, "spaces.py"), "w", encoding="utf-8") as f:
        f.write("def GPU(fn=None, **kwargs):\n    return fn if fn is not None else (lambda f: f)\n")

# 5. Download FBX2glTF binary for fast FBX -> animated GLB conversion
print("[5/5] Ensuring FBX2glTF binary is installed...")
fbx2gltf_bin = os.path.join(MIA_REPO_DIR, "util", "FBX2glTF")
os.makedirs(os.path.dirname(fbx2gltf_bin), exist_ok=True)
if not os.path.isfile(fbx2gltf_bin) or os.path.getsize(fbx2gltf_bin) < 100_000:
    try:
        subprocess.run(
            ["wget", "-q", "https://github.com/facebookincubator/FBX2glTF/releases/download/v0.9.7/FBX2glTF-linux-x64", "-O", fbx2gltf_bin],
            check=True
        )
        os.chmod(fbx2gltf_bin, 0o755)
    except Exception as e:
        print(f"[FBX2glTF] Note: {e} (will fall back to Blender native glTF exporter)")

print(f"[OK] Step 1 complete in {time.time() - t0:.1f}s. Repo at {MIA_REPO_DIR}")


In [ ]:
# @title STEP 2 — Download Pretrained MIA v2 & v1 Weights + Mixamo Motion Library
# @markdown Downloads and caches the official pretrained checkpoints from [`jasongzy/Make-It-Animatable`](https://huggingface.co/jasongzy/Make-It-Animatable) (`output/best/v2/*` and `output/best/new/*`), example character assets (`data/**`), and sets up the Mixamo template skeleton (`data/Mixamo/bones.fbx`) plus a 7-clip built-in Mixamo locomotion library (even without a gated `HF_TOKEN`!).

import os, sys, glob, time, shutil, subprocess
from huggingface_hub import snapshot_download

HF_TOKEN = ""  # @param {type:"string"}
DOWNLOAD_V1_3DGS_WEIGHTS = True  # @param {type:"boolean"}

t0 = time.time()
token_arg = HF_TOKEN.strip() if HF_TOKEN.strip() else os.environ.get("HF_TOKEN")

weights_cache_dir = os.path.join(DRIVE_CACHE_DIR, "mia_weights")
os.makedirs(weights_cache_dir, exist_ok=True)

allow_patterns = [
    "output/best/v2/**",
    "data/**",
]
if DOWNLOAD_V1_3DGS_WEIGHTS:
    allow_patterns.append("output/best/new/**")

print("[1/3] Downloading Make-It-Animatable checkpoints & example assets to Drive cache...")
snapshot_download(
    repo_id="jasongzy/Make-It-Animatable",
    repo_type="model",
    local_dir=weights_cache_dir,
    allow_patterns=allow_patterns,
    token=token_arg,
)

# Link or copy output/ and data/ into MIA_REPO_DIR
for sub in ["output/best/v2", "output/best/new", "data/examples"]:
    src_d = os.path.join(weights_cache_dir, sub)
    dst_d = os.path.join(MIA_REPO_DIR, sub)
    if os.path.isdir(src_d):
        os.makedirs(os.path.dirname(dst_d), exist_ok=True)
        if not os.path.exists(dst_d):
            try:
                os.symlink(src_d, dst_d)
            except OSError:
                shutil.copytree(src_d, dst_d, dirs_exist_ok=True)

std_run_src = os.path.join(weights_cache_dir, "data", "Standard Run.fbx")
std_run_dst = os.path.join(MIA_REPO_DIR, "data", "Standard Run.fbx")
if os.path.isfile(std_run_src) and not os.path.isfile(std_run_dst):
    os.makedirs(os.path.dirname(std_run_dst), exist_ok=True)
    shutil.copy2(std_run_src, std_run_dst)

# 2. Setup data/Mixamo/bones.fbx and animation library
mixamo_dir = os.path.join(MIA_REPO_DIR, "data", "Mixamo")
mixamo_anim_dir = os.path.join(mixamo_dir, "animation")
os.makedirs(mixamo_anim_dir, exist_ok=True)

if token_arg:
    print("[2/3] Attempting optional download of extra animations from jasongzy/Mixamo...")
    try:
        snapshot_download(
            repo_id="jasongzy/Mixamo",
            repo_type="dataset",
            local_dir=mixamo_dir,
            allow_patterns=["bones*.fbx", "animation/*.fbx"],
            token=token_arg,
        )
    except Exception as e:
        print(f"  [Note] jasongzy/Mixamo extra dataset skipped ({e}); using built-in Mixamo template & animations.")

# Guaranteed zero-token fallback for data/Mixamo/bones.fbx and built-in Mixamo animation presets
bones_fbx_path = os.path.join(mixamo_dir, "bones.fbx")
if not os.path.isfile(bones_fbx_path) and os.path.isfile(std_run_dst):
    shutil.copy2(std_run_dst, bones_fbx_path)
    print("  • Populated data/Mixamo/bones.fbx from ungated Mixamo template (Standard Run.fbx).")

if os.path.isfile(std_run_dst):
    shutil.copy2(std_run_dst, os.path.join(mixamo_anim_dir, "01_Standard_Run.fbx"))

# Generate 6 additional Mixamo locomotion FBX clips using headless Blender if not already built
builtin_clips = [
    "02_Idle_Breathing.fbx",
    "03_Walk_Cycle.fbx",
    "04_Heroic_Wave.fbx",
    "05_Victory_Jump.fbx",
    "06_Combat_Guard.fbx",
    "07_Dance_Groove.fbx",
]
if os.path.isfile(bones_fbx_path) and not all(os.path.isfile(os.path.join(mixamo_anim_dir, c)) for c in builtin_clips):
    print("[2.5/3] Synthesizing 6 built-in Mixamo locomotion FBX clips via Blender (bpy)...")
    gen_script = os.path.join(MIA_REPO_DIR, "_build_builtin_mixamo_clips.py")
    with open(gen_script, "w", encoding="utf-8") as f:
        f.write(f'''import os, math
import bpy

bones_fbx = {repr(bones_fbx_path)}
anim_dir = {repr(mixamo_anim_dir)}

def make_clip(filename, n_frames, pose_fn):
    bpy.ops.wm.read_factory_settings(use_empty=True)
    bpy.ops.import_scene.fbx(filepath=bones_fbx)
    arm = next((o for o in bpy.context.scene.objects if o.type == "ARMATURE"), None)
    if arm is None:
        return
    for o in list(bpy.context.scene.objects):
        if o.type == "MESH":
            bpy.data.objects.remove(o, do_unlink=True)
    arm.animation_data_clear()
    bpy.context.view_layer.objects.active = arm
    bpy.ops.object.mode_set(mode="POSE")
    for pb in arm.pose.bones:
        pb.rotation_mode = "XYZ"
        pb.location = (0, 0, 0)
        pb.rotation_euler = (0, 0, 0)
    scn = bpy.context.scene
    scn.frame_start = 1
    scn.frame_end = n_frames
    for fr in range(1, n_frames + 1):
        scn.frame_set(fr)
        t = (fr - 1) / float(max(1, n_frames - 1)) * 2.0 * math.pi
        pose_fn(arm.pose.bones, t, fr)
        for pb in arm.pose.bones:
            pb.keyframe_insert(data_path="rotation_euler", frame=fr)
            if "Hips" in pb.name:
                pb.keyframe_insert(data_path="location", frame=fr)
    bpy.ops.object.mode_set(mode="OBJECT")
    out_p = os.path.join(anim_dir, filename)
    bpy.ops.export_scene.fbx(filepath=out_p, use_selection=False, add_leaf_bones=False, bake_anim=True)

def _b(pbs, name):
    return pbs.get("mixamorig:" + name) or pbs.get(name)

def idle_fn(pbs, t, fr):
    if _b(pbs, "Spine1"): _b(pbs, "Spine1").rotation_euler = (0.04 * math.sin(t), 0, 0.02 * math.cos(t))
    if _b(pbs, "Neck"): _b(pbs, "Neck").rotation_euler = (0.03 * math.cos(t), 0.05 * math.sin(t), 0)
    if _b(pbs, "LeftArm"): _b(pbs, "LeftArm").rotation_euler = (0.03 * math.sin(t), 0, -1.15)
    if _b(pbs, "RightArm"): _b(pbs, "RightArm").rotation_euler = (0.03 * math.sin(t), 0, 1.15)

def walk_fn(pbs, t, fr):
    if _b(pbs, "LeftUpLeg"): _b(pbs, "LeftUpLeg").rotation_euler = (0.45 * math.sin(t), 0, 0)
    if _b(pbs, "RightUpLeg"): _b(pbs, "RightUpLeg").rotation_euler = (-0.45 * math.sin(t), 0, 0)
    if _b(pbs, "LeftLeg"): _b(pbs, "LeftLeg").rotation_euler = (max(0, -0.5 * math.cos(t)), 0, 0)
    if _b(pbs, "RightLeg"): _b(pbs, "RightLeg").rotation_euler = (max(0, 0.5 * math.cos(t)), 0, 0)
    if _b(pbs, "LeftArm"): _b(pbs, "LeftArm").rotation_euler = (-0.35 * math.sin(t), 0, -1.1)
    if _b(pbs, "RightArm"): _b(pbs, "RightArm").rotation_euler = (0.35 * math.sin(t), 0, 1.1)

def wave_fn(pbs, t, fr):
    if _b(pbs, "LeftArm"): _b(pbs, "LeftArm").rotation_euler = (0, 0, -1.15)
    if _b(pbs, "RightArm"): _b(pbs, "RightArm").rotation_euler = (0, 0, -0.45)
    if _b(pbs, "RightForeArm"): _b(pbs, "RightForeArm").rotation_euler = (0, 0, -0.6 + 0.35 * math.sin(2 * t))
    if _b(pbs, "Head"): _b(pbs, "Head").rotation_euler = (0.05 * math.sin(t), -0.1, 0)

def jump_fn(pbs, t, fr):
    s = math.sin(t * 0.5)
    if _b(pbs, "LeftArm"): _b(pbs, "LeftArm").rotation_euler = (0, 0, -1.0 + 1.4 * s)
    if _b(pbs, "RightArm"): _b(pbs, "RightArm").rotation_euler = (0, 0, 1.0 - 1.4 * s)
    if _b(pbs, "LeftUpLeg"): _b(pbs, "LeftUpLeg").rotation_euler = (0.3 * s, 0, -0.1 * s)
    if _b(pbs, "RightUpLeg"): _b(pbs, "RightUpLeg").rotation_euler = (0.3 * s, 0, 0.1 * s)

def combat_fn(pbs, t, fr):
    if _b(pbs, "Spine1"): _b(pbs, "Spine1").rotation_euler = (0.18, 0.25 * math.sin(t), 0)
    if _b(pbs, "LeftArm"): _b(pbs, "LeftArm").rotation_euler = (0.5, 0, -0.6)
    if _b(pbs, "LeftForeArm"): _b(pbs, "LeftForeArm").rotation_euler = (1.1 + 0.2 * math.sin(t), 0, 0)
    if _b(pbs, "RightArm"): _b(pbs, "RightArm").rotation_euler = (0.4 + 0.6 * max(0, math.sin(t)), 0, 0.6)
    if _b(pbs, "RightForeArm"): _b(pbs, "RightForeArm").rotation_euler = (1.2 - 0.8 * max(0, math.sin(t)), 0, 0)

def dance_fn(pbs, t, fr):
    if _b(pbs, "Spine"): _b(pbs, "Spine").rotation_euler = (0.08 * math.sin(2 * t), 0.2 * math.cos(t), 0.12 * math.sin(t))
    if _b(pbs, "LeftArm"): _b(pbs, "LeftArm").rotation_euler = (0.4 * math.sin(t), 0, -0.7 + 0.4 * math.cos(2 * t))
    if _b(pbs, "RightArm"): _b(pbs, "RightArm").rotation_euler = (-0.4 * math.sin(t), 0, 0.7 - 0.4 * math.cos(2 * t))
    if _b(pbs, "LeftUpLeg"): _b(pbs, "LeftUpLeg").rotation_euler = (0.25 * math.sin(t), 0, 0)
    if _b(pbs, "RightUpLeg"): _b(pbs, "RightUpLeg").rotation_euler = (-0.25 * math.sin(t), 0, 0)

make_clip("02_Idle_Breathing.fbx", 48, idle_fn)
make_clip("03_Walk_Cycle.fbx", 40, walk_fn)
make_clip("04_Heroic_Wave.fbx", 48, wave_fn)
make_clip("05_Victory_Jump.fbx", 40, jump_fn)
make_clip("06_Combat_Guard.fbx", 44, combat_fn)
make_clip("07_Dance_Groove.fbx", 60, dance_fn)
''')
    subprocess.run([sys.executable, gen_script], check=False)

# 3. Scan Google Drive AEI_3D_Out for 3D character meshes & 3DGS PLY files
def scan_drive_3d_assets() -> dict:
    roots = [
        os.path.join(MIA_REPO_DIR, "data", "examples"),
        "/content/drive/MyDrive/AEI_3D_Out",
        OUTPUT_BASE_DIR,
    ]
    meshes, splats = [], []
    seen = set()
    for r in roots:
        if not os.path.isdir(r):
            continue
        for dirpath, _, filenames in os.walk(r):
            if "/_thumbs" in dirpath:
                continue
            for fn in filenames:
                full = os.path.join(dirpath, fn)
                if full in seen:
                    continue
                seen.add(full)
                ext = os.path.splitext(fn)[1].lower()
                if ext in {".glb", ".obj", ".fbx"}:
                    meshes.append(full)
                elif ext == ".ply":
                    if "lgm" in fn.lower() or "splat" in full.lower() or "3dgs" in full.lower() or "lhm" in full.lower():
                        splats.append(full)
                    else:
                        meshes.append(full)
    anims = sorted(glob.glob(os.path.join(mixamo_anim_dir, "*.fbx")))
    return {"meshes": meshes[:100], "splats": splats[:60], "animations": anims}

DRIVE_3D_ASSETS = scan_drive_3d_assets()
print(f"[OK] Step 2 complete in {time.time() - t0:.1f}s")
print(f"     TEMPLATE_PATH        : {bones_fbx_path} (exists: {os.path.isfile(bones_fbx_path)})")
print(f"     Mixamo Animations    : {len(DRIVE_3D_ASSETS['animations'])} clips ready")
print(f"     Example & Drive 3D   : {len(DRIVE_3D_ASSETS['meshes'])} meshes, {len(DRIVE_3D_ASSETS['splats'])} 3DGS .ply files")


In [ ]:
# @title STEP 3 — Core Auto-Rigging, Skinning & Blender Motion Retargeting Engine
# @markdown Loads the **MIA v2** (`Hunyuan3D-2.1 ShapeVAE`) and **MIA v1** (`3DGS / PointCloud`) models and wraps the coarse joint localization, 32,768-point surface/hand sampler, k-ring weight post-processor, `ortho6d` Pose-to-Rest solver, and isolated `app_blender.py` + `FBX2glTF` retargeting pipeline.

import os, sys, gc, time, glob, zipfile, tempfile, subprocess
from pathlib import Path
import numpy as np
import torch
import torch.nn.functional as F
import trimesh

if MIA_REPO_DIR not in sys.path:
    sys.path.insert(0, MIA_REPO_DIR)

os.chdir(MIA_REPO_DIR)

from model import PCAE
from util.dataset_mixamo import (
    BONES_IDX_DICT,
    JOINTS_NUM,
    KINEMATIC_TREE,
    MIXAMO_PREFIX,
    TEMPLATE_PATH,
    get_hips_transform,
)
from util.utils import (
    fix_random,
    get_normalize_transform,
    load_gs,
    save_gs,
    sample_mesh,
    transform_gs,
    pose_local_to_global,
    pose_rot_to_global,
    to_pose_local,
    to_pose_matrix,
)
from pytorch3d.transforms import Transform3d
import app_v2 as mia_v2

MIA_V2_LOADED = False
MIA_V1_MODELS = None
CACHED_RIG_STATE = {"db": None, "mesh_name": None}

def ensure_mia_v2_loaded():
    """Lazy-loads MIA v2 checkpoints (Hunyuan3D-2.1 ShapeVAE backbone) onto GPU."""
    global MIA_V2_LOADED
    if MIA_V2_LOADED and getattr(mia_v2, "model_bw", None) is not None:
        return
    t0 = time.time()
    print("[MIA v2] Loading Hunyuan3D-2.1 ShapeVAE rigging, skinning & pose models...")
    mia_v2.init_models()
    MIA_V2_LOADED = True
    print(f"[MIA v2] Loaded on {mia_v2.device} in {time.time() - t0:.1f}s")

def run_blender_rig_and_retarget_subprocess(
    db,
    output_fbx_or_blend: str,
    rest_glb_path: str | None = None,
    reset_to_rest: bool = True,
    remove_fingers: bool = False,
    rest_pose_type: str = "No",
    ignore_pose_parts: list[str] | None = None,
    animation_file: str | None = None,
    retarget: bool = True,
    inplace: bool = True,
    restore_global: bool = False,
) -> tuple[str, str | None, str | None]:
    """
    Serializes the predicted rig state to a temporary .npz file and runs `app_blender.py`
    in a clean subprocess (avoiding main-thread locks in Colab), then converts the output
    .fbx to an interactive animated .glb via FBX2glTF (with Blender glTF export fallback).
    """
    pose_ignore_list = mia_v2.get_pose_ignore_list(rest_pose_type, ignore_pose_parts or [])
    gs_val = db.gs_rest if reset_to_rest else db.gs
    if isinstance(gs_val, torch.Tensor):
        gs_val = gs_val.cpu().numpy()

    tmp_npz = tempfile.mktemp(suffix="_mia_rig.npz")
    np.savez(
        tmp_npz,
        mesh=np.array(db.mesh, dtype=object),
        gs=gs_val,
        joints=db.joints,
        joints_tail=db.joints_tail,
        bw=db.bw,
        pose=db.pose,
        bones_idx_dict=np.array(dict(BONES_IDX_DICT), dtype=object),
        pose_ignore_list=np.array(pose_ignore_list, dtype=object),
    )

    try:
        cmd = [
            sys.executable, os.path.join(MIA_REPO_DIR, "app_blender.py"),
            "--input_path", tmp_npz,
            "--output_path", output_fbx_or_blend,
            "--template_path", TEMPLATE_PATH,
        ]
        if restore_global:
            cmd.append("--keep_raw")
        if rest_glb_path and db.is_mesh:
            cmd.extend(["--rest_path", rest_glb_path])
        if reset_to_rest:
            cmd.append("--reset_to_rest")
        if remove_fingers:
            cmd.append("--remove_fingers")
        if animation_file and os.path.isfile(animation_file):
            cmd.extend(["--animation_path", animation_file])
        if retarget:
            cmd.append("--retarget")
        if inplace:
            cmd.append("--inplace")

        r = subprocess.run(cmd, cwd=MIA_REPO_DIR, capture_output=True, text=True, check=False)
        if r.returncode != 0 or not os.path.isfile(output_fbx_or_blend):
            raise RuntimeError(f"Blender rigging/retargeting failed:\n{r.stderr[-1000:]}")

        anim_glb_path = None
        if db.is_mesh and output_fbx_or_blend.endswith(".fbx"):
            anim_glb_path = os.path.splitext(output_fbx_or_blend)[0] + "_anim.glb"
            fbx2gltf_bin = os.path.join(MIA_REPO_DIR, "util", "FBX2glTF")
            if os.path.isfile(fbx2gltf_bin):
                with tempfile.TemporaryDirectory() as tmpd:
                    subprocess.run(
                        [fbx2gltf_bin, "--binary", "--keep-attribute", "auto", "--fbx-temp-dir", tmpd,
                         "--input", os.path.abspath(output_fbx_or_blend), "--output", os.path.abspath(anim_glb_path)],
                        capture_output=True, text=True, check=False
                    )
            if not os.path.isfile(anim_glb_path):
                # Fallback: convert FBX -> animated GLB via headless bpy
                conv_code = (
                    f"import bpy; bpy.ops.wm.read_factory_settings(use_empty=True); "
                    f"bpy.ops.import_scene.fbx(filepath={repr(os.path.abspath(output_fbx_or_blend))}); "
                    f"bpy.ops.export_scene.gltf(filepath={repr(os.path.abspath(anim_glb_path))}, export_format='GLB', export_animations=True)"
                )
                subprocess.run([sys.executable, "-c", conv_code], capture_output=True, text=True, check=False)

        return output_fbx_or_blend, anim_glb_path, rest_glb_path
    finally:
        if os.path.isfile(tmp_npz):
            try:
                os.remove(tmp_npz)
            except OSError:
                pass

def rig_and_animate_character_v2(
    mesh_path: str,
    no_fingers: bool = False,
    rest_pose_type: str = "No",
    ignore_pose_parts: list[str] | None = None,
    bw_fix: bool = True,
    bw_vis_bone: str = "LeftArm",
    restore_global: bool = False,
    reset_to_rest: bool = True,
    animation_file: str | None = None,
    retarget: bool = True,
    inplace: bool = True,
    output_dir: str = OUTPUT_BASE_DIR,
) -> dict:
    """
    Full MIA v2 pipeline: Coarse joint localization -> Hips normalization ->
    Fine bone, skinning weight & ortho6d pose prediction -> k-ring weight cleanup ->
    Blender Mixamo armature binding & motion retargeting -> .fbx + .glb export.
    """
    if not mesh_path or not os.path.isfile(mesh_path):
        raise ValueError("Please upload or select a valid 3D character mesh (.glb, .obj, .ply).")

    ensure_mia_v2_loaded()
    t0 = time.time()
    slug = Path(mesh_path).stem[:32]
    ts = time.strftime("%Y%m%d_%H%M%S")
    job_dir = os.path.join(output_dir, f"{slug}_{ts}")
    os.makedirs(job_dir, exist_ok=True)

    db = mia_v2.DB()
    mia_v2.fix_random()

    # Step A: Load & sample mesh
    mia_v2.prepare_input(mesh_path, is_gs=False, opacity_threshold=0.0, db=db, export_temp=False)
    db.output_dir = job_dir
    db.joints_coarse_path = os.path.join(job_dir, f"{slug}_joints_coarse.glb")
    db.normed_path = os.path.join(job_dir, f"{slug}_normed.glb")
    db.sample_path = os.path.join(job_dir, f"{slug}_sample.glb")
    db.bw_path = os.path.join(job_dir, f"{slug}_weights_{bw_vis_bone}.glb")
    db.joints_path = os.path.join(job_dir, f"{slug}_skeleton.glb")
    db.rest_lbs_path = os.path.join(job_dir, f"{slug}_rest_lbs.glb")
    db.rest_vis_path = os.path.join(job_dir, f"{slug}_rest_rigged.glb")
    db.anim_path = os.path.join(job_dir, f"{slug}_animated.fbx")
    db.anim_vis_path = os.path.join(job_dir, f"{slug}_animated.glb")

    # Step B: Coarse joint localization & Hips coordinate alignment
    mia_v2.preprocess(db)

    # Step C: Fine bone, blend-weight & pose-to-rest inference
    t_inf0 = time.time()
    mia_v2.infer(input_normal=True, db=db)
    infer_ms = (time.time() - t_inf0) * 1000.0

    # Step D: k-ring topological weight post-processing & 3D previews
    mia_v2.vis(bw_fix=bw_fix, bw_vis_bone=bw_vis_bone, no_fingers=no_fingers, restore_global=restore_global, db=db)

    # Cache the solved rig state for instant multi-animation retargeting in Tab 2
    CACHED_RIG_STATE["db"] = db
    CACHED_RIG_STATE["mesh_name"] = slug

    # Step E: Blender armature binding + Mixamo animation retargeting
    fbx_out, anim_glb, rest_glb = run_blender_rig_and_retarget_subprocess(
        db=db,
        output_fbx_or_blend=db.anim_path,
        rest_glb_path=db.rest_vis_path,
        reset_to_rest=reset_to_rest,
        remove_fingers=no_fingers,
        rest_pose_type=rest_pose_type,
        ignore_pose_parts=ignore_pose_parts or [],
        animation_file=animation_file,
        retarget=retarget,
        inplace=inplace,
        restore_global=restore_global,
    )

    return {
        "skeleton_glb": db.joints_path if os.path.isfile(db.joints_path) else db.joints_coarse_path,
        "weights_glb": db.bw_path if os.path.isfile(db.bw_path) else None,
        "rest_glb": rest_glb if (rest_glb and os.path.isfile(rest_glb)) else db.rest_lbs_path,
        "anim_glb": anim_glb if (anim_glb and os.path.isfile(anim_glb)) else rest_glb,
        "anim_fbx": fbx_out,
        "infer_ms": round(infer_ms, 1),
        "total_sec": round(time.time() - t0, 2),
        "job_dir": job_dir,
    }

def export_locomotion_pack_zip(
    animation_files: list[str],
    no_fingers: bool = False,
    reset_to_rest: bool = True,
    retarget: bool = True,
    inplace: bool = True,
    output_dir: str = OUTPUT_BASE_DIR,
) -> tuple[str, str | None, str]:
    """
    Batch-retargets multiple Mixamo .fbx motion clips onto the currently cached character rig
    (from Tab 1) without re-running neural inference, bundling all animated .glb and .fbx files into a .zip!
    """
    db = CACHED_RIG_STATE.get("db")
    slug = CACHED_RIG_STATE.get("mesh_name") or "character"
    if db is None or db.mesh is None or db.joints is None or db.bw is None:
        raise ValueError("No cached character rig found! Please rig a character in Tab 1 first.")

    valid_anims = [a for a in animation_files if a and os.path.isfile(a)]
    if not valid_anims:
        raise ValueError("Please select at least 1 Mixamo animation clip to include in the locomotion pack.")

    ts = time.strftime("%Y%m%d_%H%M%S")
    pack_dir = os.path.join(output_dir, f"{slug}_locomotion_pack_{ts}")
    os.makedirs(pack_dir, exist_ok=True)

    exported_files = []
    preview_glb = None
    for idx, anim_p in enumerate(valid_anims, 1):
        anim_name = Path(anim_p).stem
        out_fbx = os.path.join(pack_dir, f"{slug}_{anim_name}.fbx")
        out_rest = os.path.join(pack_dir, f"{slug}_T_Pose_Rest.glb") if idx == 1 else None
        fbx_f, glb_f, rest_f = run_blender_rig_and_retarget_subprocess(
            db=db,
            output_fbx_or_blend=out_fbx,
            rest_glb_path=out_rest,
            reset_to_rest=reset_to_rest,
            remove_fingers=no_fingers,
            animation_file=anim_p,
            retarget=retarget,
            inplace=inplace,
        )
        if os.path.isfile(fbx_f):
            exported_files.append(fbx_f)
        if glb_f and os.path.isfile(glb_f):
            exported_files.append(glb_f)
            if preview_glb is None:
                preview_glb = glb_f
        if rest_f and os.path.isfile(rest_f):
            exported_files.append(rest_f)

    zip_path = os.path.join(output_dir, f"{slug}_GameEngine_Locomotion_Pack_{ts}.zip")
    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        for fp in exported_files:
            zf.write(fp, arcname=os.path.basename(fp))

    summary = (
        f"✅ **Exported {len(valid_anims)}-Clip Locomotion Pack for `{slug}`!**\n"
        f"- **ZIP Archive**: `{zip_path}` ({len(exported_files)} `.fbx` & `.glb` files)\n"
        f"- Ready for direct import into **Unity**, **Unreal Engine 5**, **Godot 4**, or **Blender**."
    )
    return zip_path, preview_glb, summary

def ensure_mia_v1_gs_loaded():
    """Lazy-loads MIA v1 point-cloud/3DGS models (input_normal=False) from output/best/new/."""
    global MIA_V1_MODELS
    if MIA_V1_MODELS is not None:
        return MIA_V1_MODELS
    t0 = time.time()
    dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    N = 32768
    h_ratio = 0.5
    m_coarse = PCAE(N=N, input_normal=False, deterministic=True, output_dim=JOINTS_NUM, predict_bw=False, predict_joints=True, predict_joints_tail=True)
    m_coarse.load("output/best/new/joints_coarse.pth")
    m_coarse.to(dev).eval()

    m_bw = PCAE(N=N, input_normal=False, deterministic=True, hierarchical_ratio=h_ratio, output_dim=JOINTS_NUM)
    m_bw.load("output/best/new/bw.pth")
    m_bw.to(dev).eval()

    m_joints = PCAE(N=N, input_normal=False, deterministic=True, hierarchical_ratio=h_ratio, output_dim=JOINTS_NUM, kinematic_tree=KINEMATIC_TREE, predict_bw=False, predict_joints=True, predict_joints_tail=True, joints_attn_causal=True)
    m_joints.load("output/best/new/joints.pth")
    m_joints.to(dev).eval()

    m_pose = PCAE(N=N, input_normal=False, deterministic=True, hierarchical_ratio=h_ratio, output_dim=JOINTS_NUM, kinematic_tree=KINEMATIC_TREE, predict_bw=False, predict_pose_trans=True, pose_mode="ortho6d", pose_input_joints=True, pose_attn_causal=True)
    m_pose.load("output/best/new/pose.pth")
    m_pose.to(dev).eval()

    MIA_V1_MODELS = {"device": dev, "coarse": m_coarse, "bw": m_bw, "joints": m_joints, "pose": m_pose}
    print(f"[MIA v1 3DGS] Loaded point-cloud/3DGS rigging models in {time.time() - t0:.1f}s")
    return MIA_V1_MODELS

@torch.no_grad()
def rig_gaussian_splatting_v1(
    ply_path: str,
    opacity_threshold: float = 0.01,
    no_fingers: bool = False,
    bw_fix: bool = True,
    reset_to_rest: bool = True,
    animation_file: str | None = None,
    output_dir: str = OUTPUT_BASE_DIR,
) -> dict:
    """Rigs a 3D Gaussian Splatting (.ply) character using MIA v1 (input_normal=False) and exports a rigged .blend + rest .ply + skeleton .glb."""
    if not ply_path or not os.path.isfile(ply_path):
        raise ValueError("Please upload or select a valid 3DGS .ply file.")
    m1 = ensure_mia_v1_gs_loaded()
    dev = m1["device"]
    slug = Path(ply_path).stem[:32]
    ts = time.strftime("%Y%m%d_%H%M%S")
    job_dir = os.path.join(output_dir, f"{slug}_3dgs_{ts}")
    os.makedirs(job_dir, exist_ok=True)

    db = mia_v2.DB()
    mia_v2.fix_random()
    mia_v2.prepare_input(ply_path, is_gs=True, opacity_threshold=float(opacity_threshold), db=db, export_temp=False)
    db.output_dir = job_dir
    db.joints_coarse_path = os.path.join(job_dir, f"{slug}_3dgs_coarse.glb")
    db.normed_path = os.path.join(job_dir, f"{slug}_normed.ply")
    db.sample_path = os.path.join(job_dir, f"{slug}_sample.glb")
    db.bw_path = os.path.join(job_dir, f"{slug}_weights.glb")
    db.joints_path = os.path.join(job_dir, f"{slug}_skeleton.glb")
    db.rest_lbs_path = os.path.join(job_dir, f"{slug}_rest_lbs.ply")
    db.rest_vis_path = None
    db.anim_path = os.path.join(job_dir, f"{slug}_3dgs_rigged.blend")
    db.anim_vis_path = None

    # Coarse localization
    norm = get_normalize_transform(db.pts, keep_ratio=True, recenter=True)
    pts = norm.transform_points(db.pts)
    verts = norm.transform_points(db.verts)
    joints_c = m1["coarse"].forward(pts.to(dev)).joints.float().cpu()
    j_head, j_tail = joints_c[..., :3], joints_c[..., 3:]
    hips = j_head[:, BONES_IDX_DICT[f"{MIXAMO_PREFIX}Hips"]]
    r_leg = j_head[:, BONES_IDX_DICT[f"{MIXAMO_PREFIX}RightUpLeg"]]
    l_leg = j_head[:, BONES_IDX_DICT[f"{MIXAMO_PREFIX}LeftUpLeg"]]
    rotate = Transform3d(matrix=get_hips_transform(hips, r_leg, l_leg).transpose(-1, -2))
    global_tf = norm.compose(rotate)
    pts = rotate.transform_points(pts)
    verts = rotate.transform_points(verts)
    db.gs = transform_gs(db.gs, global_tf)

    # Fine prediction
    norm2 = get_normalize_transform(pts, keep_ratio=True, recenter=False)
    pts = norm2.transform_points(pts)
    verts = norm2.transform_points(verts)
    db.gs = transform_gs(db.gs, norm2)
    db.pts = pts
    db.verts = verts
    db.global_transform = global_tf.compose(norm2)

    bw_chunks = [m1["bw"].forward(pts.to(dev), vc.to(dev)).bw.float().cpu() for vc in torch.split(verts, 100000, dim=-2)]
    db.bw = torch.cat(bw_chunks, dim=-2)
    db.joints = m1["joints"].forward(pts.to(dev)).joints.float().cpu()
    db.pose = m1["pose"].forward(pts.to(dev), joints=db.joints.to(dev)).pose_trans.float().cpu()

    mia_v2.vis(bw_fix=bw_fix, bw_vis_bone="LeftArm", no_fingers=no_fingers, restore_global=False, db=db)
    return {
        "skeleton_glb": db.joints_path,
        "weights_glb": db.bw_path,
        "rest_ply": db.rest_lbs_path,
        "job_dir": job_dir,
    }

print("[OK] Step 3 Core Auto-Rigging & Motion Retargeting Engine ready.")


In [ ]:
# @title STEP 4 — Launch Make-It-Animatable Gradio Studio UI
# @markdown Launches the 4-tab Gradio 5.49.1 Studio:
# @markdown 1. **💃 MIA v2 Character Auto-Rigging & Mixamo Animation Studio (`Hunyuan3D-2.1 ShapeVAE`)**
# @markdown 2. **🎮 Game Engine Multi-Animation Locomotion Pack Exporter (`.glb` + `.fbx` Bundle)**
# @markdown 3. **🦴 Interactive Bone Weight Heatmap Explorer (`52` Mixamo Bones)**
# @markdown 4. **📦 Batch Folder Character Auto-Rigger**

from IPython.display import clear_output
clear_output(wait=True)

import os, glob, traceback
import gradio as gr

BONE_NAMES_CLEAN = [n.lstrip(MIXAMO_PREFIX) for n in BONES_IDX_DICT.keys()]

def _get_anim_choices():
    anim_dir = os.path.join(MIA_REPO_DIR, "data", "Mixamo", "animation")
    files = sorted(glob.glob(os.path.join(anim_dir, "*.fbx")))
    return [(os.path.basename(f), f) for f in files]

def _refresh_assets():
    global DRIVE_3D_ASSETS
    DRIVE_3D_ASSETS = scan_drive_3d_assets()
    anim_choices = _get_anim_choices()
    return (
        gr.Dropdown(choices=[""] + DRIVE_3D_ASSETS["meshes"], value=DRIVE_3D_ASSETS["meshes"][0] if DRIVE_3D_ASSETS["meshes"] else ""),
        gr.Dropdown(choices=[("None (Static Rest Pose)", "")] + anim_choices, value=anim_choices[0][1] if anim_choices else ""),
        gr.Dropdown(choices=anim_choices, value=[c[1] for c in anim_choices[:3]]),
        f"✅ Indexed **{len(DRIVE_3D_ASSETS['meshes'])}** 3D meshes and **{len(anim_choices)}** Mixamo `.fbx` animations.",
    )

def ui_run_mia_v2(
    upload_mesh, drive_mesh,
    no_fingers, rest_pose_type, ignore_pose_parts,
    bw_fix, bw_vis_bone, restore_global,
    reset_to_rest, preset_anim, upload_anim, retarget, inplace,
):
    try:
        mesh_p = upload_mesh or (drive_mesh if drive_mesh else None)
        anim_p = upload_anim or (preset_anim if preset_anim else None)
        res = rig_and_animate_character_v2(
            mesh_path=mesh_p,
            no_fingers=bool(no_fingers),
            rest_pose_type=rest_pose_type,
            ignore_pose_parts=ignore_pose_parts or [],
            bw_fix=bool(bw_fix),
            bw_vis_bone=bw_vis_bone,
            restore_global=bool(restore_global),
            reset_to_rest=bool(reset_to_rest),
            animation_file=anim_p,
            retarget=bool(retarget),
            inplace=bool(inplace),
        )
        status = (
            f"✅ **Rigging & Animation Complete in `{res['total_sec']}s` (Neural Inference: `{res['infer_ms']} ms`)!**\n"
            f"- **Animated GLB**: `{res['anim_glb']}`\n"
            f"- **Animated FBX**: `{res['anim_fbx']}`\n"
            f"- **Rest T-Pose GLB**: `{res['rest_glb']}`"
        )
        return (
            res["anim_glb"],
            res["rest_glb"],
            res["skeleton_glb"],
            res["weights_glb"],
            res["anim_fbx"],
            res["anim_glb"],
            status,
        )
    except Exception as e:
        traceback.print_exc()
        return None, None, None, None, None, None, f"❌ **Error**: {e}"

def ui_inspect_bone_weight(bone_name, bw_fix, no_fingers, restore_global):
    try:
        db = CACHED_RIG_STATE.get("db")
        if db is None or db.bw is None:
            raise ValueError("Please rig a character in Tab 1 first to inspect per-bone skinning weights.")
        db.bw_path = os.path.join(db.output_dir, f"weights_{bone_name}.glb")
        mia_v2.vis(bw_fix=bool(bw_fix), bw_vis_bone=bone_name, no_fingers=bool(no_fingers), restore_global=bool(restore_global), db=db)
        return db.bw_path, f"✅ Displaying skinning weight heatmap for **`mixamorig:{bone_name}`**."
    except Exception as e:
        traceback.print_exc()
        return None, f"❌ **Error**: {e}"

def ui_export_locomotion_pack(selected_anims, upload_extra_anims, no_fingers, reset_to_rest, retarget, inplace):
    try:
        anims = list(selected_anims or [])
        if upload_extra_anims:
            for f in upload_extra_anims:
                anims.append(f if isinstance(f, str) else getattr(f, "name", str(f)))
        zip_p, prev_glb, msg = export_locomotion_pack_zip(
            animation_files=anims,
            no_fingers=bool(no_fingers),
            reset_to_rest=bool(reset_to_rest),
            retarget=bool(retarget),
            inplace=bool(inplace),
        )
        return prev_glb, zip_p, msg
    except Exception as e:
        traceback.print_exc()
        return None, None, f"❌ **Locomotion Pack Error**: {e}"

def ui_rig_3dgs(upload_ply, drive_ply, opacity_thr, no_fingers, bw_fix):
    try:
        ply_p = upload_ply or (drive_ply if drive_ply else None)
        res = rig_gaussian_splatting_v1(
            ply_path=ply_p,
            opacity_threshold=float(opacity_thr),
            no_fingers=bool(no_fingers),
            bw_fix=bool(bw_fix),
        )
        msg = (
            f"✅ **3D Gaussian Splatting (`.ply`) Rigged via MIA v1!**\n"
            f"- **Rest-Pose LBS `.ply`**: `{res['rest_ply']}`\n"
            f"- **Skeleton `.glb`**: `{res['skeleton_glb']}`"
        )
        return res["skeleton_glb"], res["weights_glb"], res["rest_ply"], msg
    except Exception as e:
        traceback.print_exc()
        return None, None, None, f"❌ **3DGS Rigging Error**: {e}"

def ui_batch_rig_folder(folder_path, preset_anim, no_fingers, reset_to_rest):
    try:
        if not folder_path or not os.path.isdir(folder_path):
            raise ValueError(f"Directory not found: {folder_path}")
        meshes = []
        for ext in ("*.glb", "*.obj", "*.ply"):
            meshes.extend(glob.glob(os.path.join(folder_path, ext)))
        meshes = sorted(meshes)[:20]
        if not meshes:
            raise ValueError(f"No .glb/.obj/.ply meshes found in {folder_path}")
        lines = [f"### Batch Rigged {len(meshes)} Mesh(es):"]
        last_glb = None
        for idx, mp in enumerate(meshes, 1):
            r = rig_and_animate_character_v2(
                mesh_path=mp,
                no_fingers=bool(no_fingers),
                reset_to_rest=bool(reset_to_rest),
                animation_file=preset_anim if preset_anim else None,
            )
            last_glb = r["anim_glb"]
            lines.append(f"{idx}. **`{os.path.basename(mp)}`** $\rightarrow$ `{r['anim_glb']}` (`{r['infer_ms']} ms` inference)")
        return last_glb, "\n".join(lines)
    except Exception as e:
        traceback.print_exc()
        return None, f"❌ **Batch Error**: {e}"

anim_choices_init = _get_anim_choices()

with gr.Blocks(title="Make-It-Animatable v2 Studio", theme=gr.themes.Soft()) as demo:
    gr.Markdown(
        "# 💃 Make-It-Animatable (MIA v2) — Sub-Second 3D Character Auto-Rigging, Skinning & Mixamo Motion Studio\n"
        "Convert static 3D characters from `TRELLIS-2`, `Hunyuan3D-2.1`, `Hunyuan3D-3`, `TripoSplat`, or `Mesh_Optimizer` into "
        "**Mixamo-rigged, T-pose normalized, animated `.glb` & `.fbx` characters** in seconds."
    )

    with gr.Row():
        refresh_btn = gr.Button("🔄 Re-Scan Drive 3D Meshes & Mixamo Animations", variant="secondary")
        scan_md = gr.Markdown(
            f"📂 Ready with **{len(DRIVE_3D_ASSETS['meshes'])}** 3D meshes and **{len(anim_choices_init)}** Mixamo `.fbx` animations."
        )

    with gr.Tabs():
        # ── TAB 1: MIA v2 Character Auto-Rigging & Mixamo Animation ──────────
        with gr.TabItem("💃 1. MIA v2 Character Auto-Rigging & Mixamo Animation"):
            with gr.Row():
                with gr.Column(scale=1):
                    t1_up_mesh = gr.Model3D(label="Upload 3D Character Mesh (.glb / .obj / .ply)")
                    t1_dr_mesh = gr.Dropdown(
                        choices=[""] + DRIVE_3D_ASSETS["meshes"],
                        value=DRIVE_3D_ASSETS["meshes"][0] if DRIVE_3D_ASSETS["meshes"] else "",
                        label="Or Pick Character Mesh from Examples / Google Drive (AEI_3D_Out)",
                        info="Includes built-in examples (ironman.glb, spiderman.glb, ninja.glb, luigi.glb, bunny.glb) + your Drive outputs.",
                    )
                    with gr.Accordion("🦴 Rigging, Pose-to-Rest & Skinning Settings", open=True):
                        with gr.Row():
                            t1_no_fingers = gr.Checkbox(
                                value=False,
                                label="No Fingers (22 Body Bones vs. 52 Full Hand Bones)",
                                info="Enable if the character has mitten/fused hands or no separate fingers.",
                            )
                            t1_bw_fix = gr.Checkbox(
                                value=True,
                                label="k-Ring Topological Weight Post-Processing",
                                info="Eliminates cross-limb skinning weight bleeding via sparse 6-ring graph voting.",
                            )
                        with gr.Row():
                            t1_rest_pose = gr.Dropdown(
                                choices=["No", "T-pose", "A-pose", "大-pose"],
                                value="No",
                                label="Input Pose Hint",
                                info="Select 'No' for arbitrary poses so MIA v2 predicts the full Pose-to-Rest transformation.",
                            )
                            t1_ignore_parts = gr.CheckboxGroup(
                                choices=["Fingers", "Arms", "Legs", "Head"],
                                value=["Head"],
                                label="Preserve Original Orientation For Parts",
                                info="Parts checked here keep their input orientation during T-pose normalization.",
                            )
                        with gr.Row():
                            t1_reset_rest = gr.Checkbox(
                                value=True,
                                label="Reset Character to Canonical T-Pose Before Animating",
                                info="Recommended ON so Mixamo motion retargeting starts from a clean T-pose.",
                            )
                            t1_restore_global = gr.Checkbox(
                                value=False,
                                label="Restore Original Global Scale/Coordinates",
                                info="Keep unchecked for standard Mixamo coordinate normalization.",
                            )

                    with gr.Accordion("🎬 Mixamo Animation & Motion Retargeting", open=True):
                        t1_preset_anim = gr.Dropdown(
                            choices=[("None (Export Static Rigged T-Pose Only)", "")] + anim_choices_init,
                            value=anim_choices_init[0][1] if anim_choices_init else "",
                            label="Select Built-In Mixamo Animation (.fbx)",
                            info="Choose a motion clip to retarget onto the rigged character.",
                        )
                        t1_up_anim = gr.File(
                            label="Or Upload Any Custom Mixamo Animation (.fbx)",
                            file_types=[".fbx"],
                        )
                        with gr.Row():
                            t1_retarget = gr.Checkbox(value=True, label="Enable Auto-Rig Pro / Mixamo Bone Retargeting", info="Adapts bone lengths and proportions to your character.")
                            t1_inplace = gr.Checkbox(value=True, label="In-Place Animation (Lock Root Horizontal Drift)", info="Keeps walking/running characters centered at the origin.")

                    t1_vis_bone = gr.Dropdown(
                        choices=BONE_NAMES_CLEAN,
                        value="LeftArm",
                        label="Initial Bone for Skinning Weight Heatmap Preview",
                        info="Visualizes the influence heatmap of this bone on the mesh.",
                    )
                    t1_btn = gr.Button("💃 Rig, Skin & Animate Character (MIA v2)", variant="primary")

                with gr.Column(scale=1):
                    t1_out_anim_glb = gr.Model3D(label="Animated 3D Character (.glb — Click Play in Viewer Bottom-Left)")
                    with gr.Row():
                        t1_out_rest_glb = gr.Model3D(label="Rigged Canonical T-Pose (.glb)")
                        t1_out_skel_glb = gr.Model3D(label="Predicted 52-Bone Skeleton (.glb)")
                    t1_out_bw_glb = gr.Model3D(label="Per-Bone Skinning Weight Heatmap (.glb)")
                    with gr.Row():
                        t1_dl_fbx = gr.File(label="Download Animated .FBX (Unreal / Unity / Blender)")
                        t1_dl_glb = gr.File(label="Download Animated .GLB (Godot / Web / Three.js)")
                    t1_status = gr.Markdown()

        # ── TAB 2: Game Engine Multi-Animation Locomotion Pack Exporter ──────
        with gr.TabItem("🎮 2. Game Engine Multi-Animation Locomotion Pack Exporter"):
            gr.Markdown(
                "### Batch-Retarget Multiple Mixamo Motions onto Your Cached Character Rig\n"
                "Once you rig a character in **Tab 1**, its 52-bone skeleton and blend weights are cached in memory. "
                "Select any combination of animations below to export a complete **Game Engine Locomotion `.zip` Pack** (`.fbx` + `.glb`) in seconds!"
            )
            with gr.Row():
                with gr.Column(scale=1):
                    t2_anims = gr.Dropdown(
                        choices=anim_choices_init,
                        value=[c[1] for c in anim_choices_init[:4]],
                        multiselect=True,
                        label="Select Mixamo Animations to Bake into Pack",
                        info="Each selected animation is retargeted into both .fbx and .glb formats.",
                    )
                    t2_up_anims = gr.File(
                        label="Optional: Upload Additional Custom Mixamo .FBX Animations",
                        file_count="multiple",
                        file_types=[".fbx"],
                    )
                    with gr.Row():
                        t2_no_fingers = gr.Checkbox(value=False, label="No Fingers (22 Body Bones)", info="Merge finger weights into hand bone.")
                        t2_reset_rest = gr.Checkbox(value=True, label="Reset to T-Pose", info="Apply canonical T-pose before retargeting.")
                        t2_retarget = gr.Checkbox(value=True, label="Retarget Proportions", info="Enable proportional retargeting.")
                        t2_inplace = gr.Checkbox(value=True, label="In-Place Locomotion", info="Lock horizontal root motion for game controllers.")
                    t2_btn = gr.Button("📦 Bake & Export Multi-Animation Game Pack (.ZIP)", variant="primary")
                with gr.Column(scale=1):
                    t2_prev_glb = gr.Model3D(label="Preview First Animated Clip (.glb)")
                    t2_zip_file = gr.File(label="Download Game Engine Locomotion Pack (.ZIP of .GLB & .FBX)")
                    t2_status = gr.Markdown()

        # ── TAB 3: Interactive 52-Bone Weight Explorer & MIA v1 3DGS Rigger ─
        with gr.TabItem("🔮 3. 52-Bone Weight Heatmap Explorer & MIA v1 3DGS (.ply) Rigger"):
            gr.Markdown("### A. Interactive 52-Bone Skinning Weight Heatmap Explorer\nInspect the exact per-vertex skinning weights (`plasma` colormap) predicted for any of the 52 Mixamo bones without re-running inference:")
            with gr.Row():
                with gr.Column(scale=1):
                    t3_bone = gr.Dropdown(choices=BONE_NAMES_CLEAN, value="Head", label="Select Mixamo Bone to Visualize", info="Instantly renders vertex influence heatmap for the chosen bone.")
                    t3_bw_fix = gr.Checkbox(value=True, label="Enable k-Ring Topological Conflict Filter", info="Compare raw vs. k-ring filtered weights.")
                    t3_no_fingers = gr.Checkbox(value=False, label="Merge Finger Weights into Hand Bone", info="Inspect 22-bone vs. 52-bone weight distribution.")
                    t3_restore_global = gr.Checkbox(value=False, label="Restore Original Coordinates", info="Display in normalized vs. original coordinates.")
                    t3_btn = gr.Button("🔍 Render Bone Weight Heatmap", variant="primary")
                with gr.Column(scale=1):
                    t3_out_glb = gr.Model3D(label="Per-Bone Skinning Weight Heatmap (.glb)")
                    t3_status = gr.Markdown()

            gr.Markdown("---\n### B. 🔮 MIA v1 3D Gaussian Splatting (`.ply`) Character Rigger\nRig point-cloud / 3D Gaussian Splatting `.ply` characters (e.g. `boy_lgm.ply` or outputs from `TripoSplat`, `NoPoSplat`, `LHM`) using MIA v1 (`input_normal=False`):")
            with gr.Row():
                with gr.Column(scale=1):
                    t3_gs_up = gr.File(label="Upload 3D Gaussian Splatting Character (.ply)", file_types=[".ply"])
                    t3_gs_dr = gr.Dropdown(
                        choices=[""] + DRIVE_3D_ASSETS["splats"],
                        value=DRIVE_3D_ASSETS["splats"][0] if DRIVE_3D_ASSETS["splats"] else "",
                        label="Or Pick 3DGS .PLY from Examples / Drive",
                        info="Includes built-in boy_lgm.ply example + 3DGS files from AEI_3D_Out.",
                    )
                    with gr.Row():
                        t3_gs_opac = gr.Slider(0.0, 0.5, value=0.01, step=0.005, label="Gaussian Opacity Threshold", info="Filters out transparent floaters before surface point sampling.")
                        t3_gs_nofing = gr.Checkbox(value=False, label="No Fingers (22 Body Bones)", info="Recommended ON for coarse 3DGS splats.")
                        t3_gs_bwfix = gr.Checkbox(value=True, label="Weight Post-Processing", info="Applies topological weight cleanup.")
                    t3_gs_btn = gr.Button("🔮 Rig 3DGS Character (MIA v1)", variant="secondary")
                with gr.Column(scale=1):
                    with gr.Row():
                        t3_gs_skel = gr.Model3D(label="Predicted 3DGS Skeleton (.glb)")
                        t3_gs_bw = gr.Model3D(label="Predicted 3DGS Weight Cloud (.glb)")
                    t3_gs_ply = gr.File(label="Download Rest-Pose Transformed 3DGS (.ply)")
                    t3_gs_status = gr.Markdown()

        # ── TAB 4: Batch Folder Auto-Rigger ──────────────────────────────────
        with gr.TabItem("📦 4. Batch Folder Character Auto-Rigger"):
            with gr.Row():
                with gr.Column(scale=1):
                    t4_dir = gr.Textbox(
                        value=os.path.join(MIA_REPO_DIR, "data", "examples"),
                        label="Input Folder of Static Character Meshes (.glb / .obj / .ply)",
                        info="Can be any folder on Google Drive (e.g. /content/drive/MyDrive/AEI_3D_Out/TRELLIS-2).",
                    )
                    t4_anim = gr.Dropdown(
                        choices=[("None (T-Pose Rig Only)", "")] + anim_choices_init,
                        value=anim_choices_init[0][1] if anim_choices_init else "",
                        label="Animation to Apply to All Characters",
                        info="Select a Mixamo animation or leave on T-Pose Rig Only.",
                    )
                    with gr.Row():
                        t4_no_fingers = gr.Checkbox(value=False, label="No Fingers", info="Use 22 body bones.")
                        t4_reset_rest = gr.Checkbox(value=True, label="Reset to T-Pose", info="Normalize input poses to T-pose.")
                    t4_btn = gr.Button("🚀 Batch Rig & Animate Folder", variant="primary")
                with gr.Column(scale=1):
                    t4_out_glb = gr.Model3D(label="Last Processed Character (.glb)")
                    t4_status = gr.Markdown()

    refresh_btn.click(
        fn=_refresh_assets,
        inputs=[],
        outputs=[t1_dr_mesh, t1_preset_anim, t2_anims, scan_md],
        concurrency_limit=1,
    )
    t1_btn.click(
        fn=ui_run_mia_v2,
        inputs=[
            t1_up_mesh, t1_dr_mesh, t1_no_fingers, t1_rest_pose, t1_ignore_parts,
            t1_bw_fix, t1_vis_bone, t1_restore_global, t1_reset_rest,
            t1_preset_anim, t1_up_anim, t1_retarget, t1_inplace,
        ],
        outputs=[t1_out_anim_glb, t1_out_rest_glb, t1_out_skel_glb, t1_out_bw_glb, t1_dl_fbx, t1_dl_glb, t1_status],
        concurrency_limit=1,
    )
    t2_btn.click(
        fn=ui_export_locomotion_pack,
        inputs=[t2_anims, t2_up_anims, t2_no_fingers, t2_reset_rest, t2_retarget, t2_inplace],
        outputs=[t2_prev_glb, t2_zip_file, t2_status],
        concurrency_limit=1,
    )
    t3_btn.click(
        fn=ui_inspect_bone_weight,
        inputs=[t3_bone, t3_bw_fix, t3_no_fingers, t3_restore_global],
        outputs=[t3_out_glb, t3_status],
        concurrency_limit=1,
    )
    t3_gs_btn.click(
        fn=ui_rig_3dgs,
        inputs=[t3_gs_up, t3_gs_dr, t3_gs_opac, t3_gs_nofing, t3_gs_bwfix],
        outputs=[t3_gs_skel, t3_gs_bw, t3_gs_ply, t3_gs_status],
        concurrency_limit=1,
    )
    t4_btn.click(
        fn=ui_batch_rig_folder,
        inputs=[t4_dir, t4_anim, t4_no_fingers, t4_reset_rest],
        outputs=[t4_out_glb, t4_status],
        concurrency_limit=1,
    )
    demo.load(
        fn=lambda: f"💃 **Make-It-Animatable v2 Studio Ready.** Output directory: `{OUTPUT_BASE_DIR}`",
        inputs=None,
        outputs=[t1_status],
    )

demo.queue(default_concurrency_limit=1).launch(share=True, inline=True, debug=False, show_error=True)


In [ ]:
# @title STEP 5 — Keep-Alive Cell (Optional)
# @markdown Keeps the Colab runtime active during batch character rigging and animation retargeting.

import time
KEEP_ALIVE_MINUTES = 60  # @param {type:"integer"}
try:
    for m in range(KEEP_ALIVE_MINUTES):
        print(f"[Keep-Alive] Active ({m + 1}/{KEEP_ALIVE_MINUTES} min)...", end="\r", flush=True)
        time.sleep(60)
except KeyboardInterrupt:
    print("\n[Keep-Alive] Stopped by user.")


In [ ]:
# @title STEP 6 — Quick Test (Rig & Animate Example Character in 1 Click)
# @markdown Runs **MIA v2** on a built-in example character (`ironman.glb` or `bunny.glb`), predicts the 52-bone skeleton + skinning weights + T-pose normalization, retargets `01_Standard_Run.fbx`, and displays download links for the animated `.glb` and `.fbx`.

import os, glob
from IPython.display import display, FileLink

example_mesh = os.path.join(MIA_REPO_DIR, "data", "examples", "ironman.glb")
if not os.path.isfile(example_mesh):
    candidates = glob.glob(os.path.join(MIA_REPO_DIR, "data", "examples", "*.glb"))
    example_mesh = candidates[0] if candidates else ""

anim_candidates = sorted(glob.glob(os.path.join(MIA_REPO_DIR, "data", "Mixamo", "animation", "*.fbx")))
default_anim = anim_candidates[0] if anim_candidates else None

print(f"[Quick Test] Rigging & animating: {example_mesh}")
print(f"             Animation clip     : {default_anim}")

res = rig_and_animate_character_v2(
    mesh_path=example_mesh,
    no_fingers=False,
    rest_pose_type="No",
    ignore_pose_parts=["Head"],
    bw_fix=True,
    bw_vis_bone="LeftArm",
    reset_to_rest=True,
    animation_file=default_anim,
    retarget=True,
    inplace=True,
)

print(f"[Quick Test] Done in {res['total_sec']}s (Inference: {res['infer_ms']} ms)")
if res.get("anim_glb") and os.path.isfile(res["anim_glb"]):
    display(FileLink(res["anim_glb"]))
if res.get("anim_fbx") and os.path.isfile(res["anim_fbx"]):
    display(FileLink(res["anim_fbx"]))
if res.get("rest_glb") and os.path.isfile(res["rest_glb"]):
    display(FileLink(res["rest_glb"]))


In [ ]:
# @title STEP 7 — Batch Rigging & Locomotion Pack Export + STEP 8 — Log / Error Inspector
# @markdown Batch-rigs all `.glb`/`.obj` character meshes in a folder and exports animated `.glb` + `.fbx` files.

import os, glob, traceback
from IPython.display import display, FileLink

BATCH_MESH_DIR = "/content/Make-It-Animatable/data/examples"  # @param {type:"string"}
BATCH_LIMIT = 3  # @param {type:"integer"}
BATCH_NO_FINGERS = False  # @param {type:"boolean"}

mesh_files = sorted(glob.glob(os.path.join(BATCH_MESH_DIR, "*.glb")))[:BATCH_LIMIT]
anim_files = sorted(glob.glob(os.path.join(MIA_REPO_DIR, "data", "Mixamo", "animation", "*.fbx")))
chosen_anim = anim_files[0] if anim_files else None

print(f"[Batch] Processing {len(mesh_files)} character mesh(es) from {BATCH_MESH_DIR}...")
for idx, mp in enumerate(mesh_files, 1):
    try:
        print(f"  [{idx}/{len(mesh_files)}] Rigging {os.path.basename(mp)}...")
        r = rig_and_animate_character_v2(
            mesh_path=mp,
            no_fingers=BATCH_NO_FINGERS,
            reset_to_rest=True,
            animation_file=chosen_anim,
            retarget=True,
            inplace=True,
        )
        print(f"     -> Animated GLB: {r['anim_glb']} ({r['infer_ms']} ms neural inference)")
        if r.get("anim_glb") and os.path.isfile(r["anim_glb"]):
            display(FileLink(r["anim_glb"]))
    except Exception as e:
        print(f"  [Error] {os.path.basename(mp)}: {e}")
        traceback.print_exc()
